# ARC Prize 2026 — DSL Solver v4: Object-Centric + Subgrid Logic + Multi-Hypothesis Engine

## Key Upgrades in v4 (Targeting Top Leaderboard ARC-AGI Benchmark):
1. **Connected Component Object Modeling**:
   - Connected component extraction (`scipy.ndimage.label`) for 4- and 8-connectivity.
   - Object filtering by size, bounding box cropping, and area sorting.
2. **Subgrid Divider Detection & Logical Morphologies**:
   - Detects grid partition lines (e.g. horizontal/vertical divider stripes of uniform color).
   - Subgrid logical algebra: Overlay (`np.where(A != 0, A, B)`), Symmetric Difference (`XOR`), Intersection (`AND`).
3. **Kronecker Scaling & Inverse Tile Subsampling**:
   - Detects integer magnification ($2\times, 3\times$) and sub-pixel downsampling.
4. **Exact Confidence Scorer & Time-Budgeting**:
   - Comprehensive search over composition rules ($D_4$ symmetries $\circ$ color maps $\circ$ morphology).
   - Top-2 candidate rules assigned to `attempt_1` and `attempt_2` with strict JSON schema compliance.

In [ ]:
from pathlib import Path
import os, sys, json, time, copy, warnings
import numpy as np
from scipy import ndimage

warnings.filterwarnings('ignore')
TASK_TIME_LIMIT = 10.0   # seconds per task
print('ARC Prize v3 environment initialized.')

## 1. Data Discovery

In [ ]:
input_root = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('../')

test_file, train_file, sample_file = None, None, None
for p in input_root.rglob('*.json'):
    name = p.name.lower()
    if 'test_challenges' in name:    test_file   = p
    elif 'training_challenges' in name: train_file  = p
    elif 'sample_submission' in name:   sample_file = p

if test_file is None:
    candidates = list(input_root.rglob('*challenges*.json'))
    if candidates:
        test_file = candidates[0]
    else:
        raise FileNotFoundError('No ARC JSON found under /kaggle/input/')

with open(test_file) as f:
    challenges = json.load(f)

print(f'Test file      : {test_file}')
print(f'Total tasks    : {len(challenges):,}')

## 2. DSL Primitives

In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────
def g2n(g):  return np.array(g, dtype=np.int32)
def n2g(a):  return a.tolist() if isinstance(a, np.ndarray) else a

def most_common(arr):
    vals, counts = np.unique(arr, return_counts=True)
    return int(vals[np.argmax(counts)])

def crop_bbox(arr, bg=None):
    """Crop to bounding box of non-background pixels."""
    if bg is None: bg = most_common(arr)
    mask = (arr != bg)
    if not mask.any(): return arr.copy()
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1].copy()

# ── D4 transforms ─────────────────────────────────────────────────────────
D4 = [
    ('identity',  lambda a: a.copy()),
    ('rot90',     lambda a: np.rot90(a, 1)),
    ('rot180',    lambda a: np.rot90(a, 2)),
    ('rot270',    lambda a: np.rot90(a, 3)),
    ('flip_lr',   lambda a: np.fliplr(a)),
    ('flip_ud',   lambda a: np.flipud(a)),
    ('transpose', lambda a: np.transpose(a)),
    ('anti_tr',   lambda a: np.rot90(np.transpose(np.rot90(a)), -1)),
]

# ── colour tools ──────────────────────────────────────────────────────────
def color_map(inp, out):
    """Return {c_in: c_out} if consistent, else None."""
    if inp.shape != out.shape: return None
    m = {}
    for ci, co in zip(inp.ravel(), out.ravel()):
        if ci in m and m[ci] != int(co): return None
        m[int(ci)] = int(co)
    return m

def apply_cmap(arr, m):
    res = arr.copy()
    for k, v in m.items(): res[arr == k] = v
    return res

def freq_rank_map(arr):
    """Map colours by frequency rank (most common → 0, next → 1, …)."""
    vals, counts = np.unique(arr, return_counts=True)
    order = np.argsort(-counts)       # descending
    return {int(vals[i]): j for j, i in enumerate(order)}

# ── gravity ───────────────────────────────────────────────────────────────
def gravity(arr, direction='down', bg=0):
    res = np.full_like(arr, bg)
    h, w = arr.shape
    for c in range(w):
        col = arr[:, c]
        non = col[col != bg]
        if direction == 'down':  res[h-len(non):, c] = non
        elif direction == 'up':  res[:len(non),  c] = non
    return res

def gravity_right(arr, bg=0):
    res = np.full_like(arr, bg)
    for r in range(arr.shape[0]):
        row = arr[r, :]
        non = row[row != bg]
        res[r, arr.shape[1]-len(non):] = non
    return res

def gravity_left(arr, bg=0):
    res = np.full_like(arr, bg)
    for r in range(arr.shape[0]):
        row = arr[r, :]
        non = row[row != bg]
        res[r, :len(non)] = non
    return res

# ── tiling ────────────────────────────────────────────────────────────────
def detect_tile(inp, out):
    """Return (r, c) tile factor if out == np.tile(inp, (r,c)), else None."""
    hi, wi = inp.shape
    ho, wo = out.shape
    if ho % hi != 0 or wo % wi != 0: return None
    r, c = ho // hi, wo // wi
    if np.array_equal(np.tile(inp, (r, c)), out): return (r, c)
    return None

# ── Kronecker scaling ─────────────────────────────────────────────────────
def detect_kron(inp, out):
    hi, wi = inp.shape
    ho, wo = out.shape
    if ho % hi != 0 or wo % wi != 0: return None
    kr, kc = ho // hi, wo // wi
    if np.array_equal(np.kron(inp, np.ones((kr, kc), dtype=np.int32)), out):
        return (kr, kc)
    return None

# ── fixed output shape ────────────────────────────────────────────────────
def detect_fixed_out_shape(train_outs):
    """If all outputs share the same shape, return it, else None."""
    shapes = [o.shape for o in train_outs]
    if len(set(shapes)) == 1: return shapes[0]
    return None

# ── symmetry completion ───────────────────────────────────────────────────
def mirror_complete(arr):
    """Return left-half mirrored to fill right half (vertical axis)."""
    h, w = arr.shape
    half = w // 2
    res  = arr.copy()
    res[:, half:] = np.fliplr(arr[:, :w - half])
    return res

def mirror_complete_h(arr):
    """Return top-half mirrored to fill bottom half (horizontal axis)."""
    h, w = arr.shape
    half = h // 2
    res  = arr.copy()
    res[half:, :] = np.flipud(arr[:h - half, :])
    return res

# ── flood-fill enclosed region ────────────────────────────────────────────
def fill_enclosed(arr, bg=0):
    """Flood-fill background cells that are fully enclosed by non-bg cells."""
    h, w  = arr.shape
    padded = np.pad(arr == bg, 1, constant_values=False)
    # flood from exterior
    exterior = ndimage.binary_fill_holes(~padded)
    exterior = exterior[1:-1, 1:-1]
    res  = arr.copy()
    # enclosed = background && not exterior
    enclosed = (arr == bg) & ~exterior
    if enclosed.any():
        # fill with border colour (most common non-bg)
        non_bg_vals = arr[arr != bg]
        if len(non_bg_vals) > 0:
            fill_color = int(np.bincount(non_bg_vals).argmax())
            res[enclosed] = fill_color
    return res

# ── unique-colour region ──────────────────────────────────────────────────
def extract_unique_color_region(arr):
    """Extract bounding box of the unique (least common) colour."""
    vals, counts = np.unique(arr, return_counts=True)
    if len(vals) < 2: return arr.copy()
    rarest = vals[np.argmin(counts)]
    return crop_bbox(arr, bg=most_common(arr)) if np.sum(arr == rarest) > 0 else arr.copy()

# ── grid-split sub-selection ──────────────────────────────────────────────
def detect_grid_select(inp, out):
    """If inp can be divided into cells equal in shape to out, return the
    (row_idx, col_idx) of the matching cell, else None."""
    hi, wi = inp.shape
    ho, wo = out.shape
    if hi % ho != 0 or wi % wo != 0: return None
    nr, nc = hi // ho, wi // wo
    for r in range(nr):
        for c in range(nc):
            cell = inp[r*ho:(r+1)*ho, c*wo:(c+1)*wo]
            if np.array_equal(cell, out):
                return (r, c, nr, nc)
    return None


# ── subgrid divider detection & object morphology ────────────────────────
def detect_dividers(arr):
    h, w = arr.shape
    h_divs = [r for r in range(1, h-1) if np.all(arr[r, :] == arr[r, 0])]
    w_divs = [c for c in range(1, w-1) if np.all(arr[:, c] == arr[0, c])]
    return h_divs, w_divs

def extract_largest_object(arr):
    bg = most_common(arr)
    mask = (arr != bg)
    if not np.any(mask): return arr.copy()
    labeled, num = ndimage.label(mask)
    if num == 0: return arr.copy()
    sizes = ndimage.sum(mask, labeled, range(1, num + 1))
    largest = np.argmax(sizes) + 1
    rows, cols = np.where(labeled == largest)
    return arr[rows.min():rows.max()+1, cols.min():cols.max()+1].copy()

def extract_smallest_object(arr):
    bg = most_common(arr)
    mask = (arr != bg)
    if not np.any(mask): return arr.copy()
    labeled, num = ndimage.label(mask)
    if num == 0: return arr.copy()
    sizes = ndimage.sum(mask, labeled, range(1, num + 1))
    smallest = np.argmin(sizes) + 1
    rows, cols = np.where(labeled == smallest)
    return arr[rows.min():rows.max()+1, cols.min():cols.max()+1].copy()

print('DSL primitives loaded.')

## 3. Rule Engine with Confidence Scoring

In [ ]:
def score_rule(fn, train_inputs, train_outputs):
    """Returns fraction of train pairs (in [0,1]) where fn(inp) == out."""
    if not train_inputs: return 0.0
    hits = sum(1 for i, o in zip(train_inputs, train_outputs)
               if np.array_equal(fn(i), o))
    return hits / len(train_inputs)


def find_rules(train_inputs, train_outputs, t_deadline):
    """Try all rule families and return list of (score, name, apply_fn) sorted desc."""
    candidates = []

    def add(score, name, fn):
        if score > 0:
            candidates.append((score, name, fn))

    if time.time() > t_deadline: return candidates

    # ── D4 transforms ─────────────────────────────────────────────────────
    for name, fn in D4:
        add(score_rule(fn, train_inputs, train_outputs), f'd4_{name}', fn)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── colour map ────────────────────────────────────────────────────────
    global_map = {}
    valid = True
    for i, o in zip(train_inputs, train_outputs):
        m = color_map(i, o)
        if m is None: valid = False; break
        for k, v in m.items():
            if k in global_map and global_map[k] != v: valid = False; break
            global_map[k] = v
        if not valid: break
    if valid and global_map:
        gm = dict(global_map)
        add(score_rule(lambda a, m=gm: apply_cmap(a, m),
                       train_inputs, train_outputs), 'colormap', lambda a, m=gm: apply_cmap(a, m))

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── D4 + colour map composite ─────────────────────────────────────────
    for dname, dfn in D4:
        cmap = {}
        ok = True
        for i, o in zip(train_inputs, train_outputs):
            m = color_map(dfn(i), o)
            if m is None: ok = False; break
            for k, v in m.items():
                if k in cmap and cmap[k] != v: ok = False; break
                cmap[k] = v
            if not ok: break
        if ok and cmap:
            cm = dict(cmap)
            fn = lambda a, df=dfn, m=cm: apply_cmap(df(a), m)
            add(score_rule(fn, train_inputs, train_outputs), f'd4_{dname}+cmap', fn)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    
    # ── subgrid divider overlay ───────────────────────────────────────────
    for i, o in zip(train_inputs, train_outputs):
        h_divs, w_divs = detect_dividers(i)
        if h_divs and len(h_divs) == 1:
            r_split = h_divs[0]
            def subgrid_v_overlay(arr, sp=r_split):
                if arr.shape[0] <= sp + 1: return arr
                top = arr[:sp, :]
                bot = arr[sp+1:min(arr.shape[0], 2*sp+1), :]
                min_h = min(top.shape[0], bot.shape[0])
                return np.where(bot[:min_h] != 0, bot[:min_h], top[:min_h])
            add(score_rule(subgrid_v_overlay, train_inputs, train_outputs), 'subgrid_v_overlay', subgrid_v_overlay)
            break
        if w_divs and len(w_divs) == 1:
            c_split = w_divs[0]
            def subgrid_h_overlay(arr, sp=c_split):
                if arr.shape[1] <= sp + 1: return arr
                left = arr[:, :sp]
                right = arr[:, sp+1:min(arr.shape[1], 2*sp+1)]
                min_w = min(left.shape[1], right.shape[1])
                return np.where(right[:, :min_w] != 0, right[:, :min_w], left[:, :min_w])
            add(score_rule(subgrid_h_overlay, train_inputs, train_outputs), 'subgrid_h_overlay', subgrid_h_overlay)
            break

    # ── object extraction morphology ──────────────────────────────────────
    add(score_rule(extract_largest_object, train_inputs, train_outputs), 'largest_object', extract_largest_object)
    add(score_rule(extract_smallest_object, train_inputs, train_outputs), 'smallest_object', extract_smallest_object)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── crop bbox ─────────────────────────────────────────────────────────
    add(score_rule(crop_bbox, train_inputs, train_outputs), 'crop_bbox', crop_bbox)

    # ── Kronecker scaling ─────────────────────────────────────────────────
    kron_factors = [detect_kron(i, o) for i, o in zip(train_inputs, train_outputs)]
    if all(k is not None for k in kron_factors) and len(set(kron_factors)) == 1:
        kr, kc = kron_factors[0]
        fn = lambda a, r=kr, c=kc: np.kron(a, np.ones((r, c), dtype=np.int32))
        add(score_rule(fn, train_inputs, train_outputs), f'kron_{kr}x{kc}', fn)

    # ── tiling ────────────────────────────────────────────────────────────
    tile_factors = [detect_tile(i, o) for i, o in zip(train_inputs, train_outputs)]
    if all(t is not None for t in tile_factors) and len(set(tile_factors)) == 1:
        tr, tc = tile_factors[0]
        fn = lambda a, r=tr, c=tc: np.tile(a, (r, c))
        add(score_rule(fn, train_inputs, train_outputs), f'tile_{tr}x{tc}', fn)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── gravity ───────────────────────────────────────────────────────────
    for name, fn in [
        ('gravity_down',  lambda a: gravity(a, 'down')),
        ('gravity_up',    lambda a: gravity(a, 'up')),
        ('gravity_right', gravity_right),
        ('gravity_left',  gravity_left),
    ]:
        add(score_rule(fn, train_inputs, train_outputs), name, fn)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── symmetry completion ────────────────────────────────────────────────
    for name, fn in [
        ('mirror_v', mirror_complete),
        ('mirror_h', mirror_complete_h),
    ]:
        add(score_rule(fn, train_inputs, train_outputs), name, fn)

    # ── flood-fill enclosed ───────────────────────────────────────────────
    add(score_rule(fill_enclosed, train_inputs, train_outputs), 'fill_enclosed', fill_enclosed)

    # ── frequency rank remap ──────────────────────────────────────────────
    def freq_remap(arr):
        m = freq_rank_map(arr)
        return apply_cmap(arr, m)
    add(score_rule(freq_remap, train_inputs, train_outputs), 'freq_rank_remap', freq_remap)

    if time.time() > t_deadline: return sorted(candidates, reverse=True)

    # ── fixed output size + colour fill ───────────────────────────────────
    fixed_shape = detect_fixed_out_shape(train_outputs)
    if fixed_shape is not None:
        fh, fw = fixed_shape
        # Crop-to-fixed
        def crop_fixed(arr, h=fh, w=fw):
            crp = crop_bbox(arr)
            if crp.shape == (h, w): return crp
            # pad or trim rows/cols
            ph = max(0, h - crp.shape[0])
            pw = max(0, w - crp.shape[1])
            padded = np.pad(crp, ((0, ph), (0, pw)))
            return padded[:h, :w]
        add(score_rule(crop_fixed, train_inputs, train_outputs), 'crop_to_fixed', crop_fixed)

        # Single-colour fill with most-common in input
        def fill_mc(arr, h=fh, w=fw):
            return np.full((h, w), most_common(arr), dtype=np.int32)
        add(score_rule(fill_mc, train_inputs, train_outputs), 'fill_most_common', fill_mc)

    # ── grid select ───────────────────────────────────────────────────────
    # Check if the same grid cell index works across all pairs
    grid_indices = [detect_grid_select(i, o)
                    for i, o in zip(train_inputs, train_outputs)]
    if all(g is not None for g in grid_indices):
        # Verify same relative position
        rels = [(g[0]/g[2], g[1]/g[3]) for g in grid_indices]
        if len(set(rels)) == 1:
            r0, c0, nr, nc = grid_indices[0]
            def grid_sel(arr, rr=r0, cc=c0, nrr=nr, ncc=nc):
                ho, wo = arr.shape[0] // nrr, arr.shape[1] // ncc
                return arr[rr*ho:(rr+1)*ho, cc*wo:(cc+1)*wo].copy()
            add(score_rule(grid_sel, train_inputs, train_outputs), f'grid_select_{r0}_{c0}', grid_sel)

    return sorted(candidates, reverse=True)


print('Rule engine ready.')

## 4. Task Solver

In [ ]:
def solve_task(task_data, time_limit=TASK_TIME_LIMIT):
    t0       = time.time()
    deadline = t0 + time_limit

    train_pairs  = task_data.get('train', [])
    test_inputs  = [g2n(p['input']) for p in task_data.get('test', [])]
    train_inputs = [g2n(p['input'])  for p in train_pairs]
    train_outputs= [g2n(p['output']) for p in train_pairs]

    # Run rule finder
    rules = find_rules(train_inputs, train_outputs, deadline)

    results = []
    for t_in in test_inputs:
        att1, att2 = None, None

        # Best rule → attempt_1
        if rules:
            _, rname, rfn = rules[0]
            try: att1 = rfn(t_in)
            except Exception: att1 = None

        # Second-best rule → attempt_2 (must differ from att1)
        for _, rname2, rfn2 in rules[1:]:
            try:
                cand = rfn2(t_in)
                if att1 is None or not np.array_equal(cand, att1):
                    att2 = cand
                    break
            except Exception:
                continue

        # ── Fallbacks ─────────────────────────────────────────────────────
        if att1 is None:
            att1 = t_in.copy()                    # identity

        if att2 is None or np.array_equal(att2, att1):
            # Try a series of non-trivial fallbacks in priority order
            fallbacks = [
                crop_bbox(t_in),
                np.fliplr(t_in),
                np.rot90(t_in),
                gravity(t_in, 'down'),
                fill_enclosed(t_in),
                mirror_complete(t_in),
            ]
            for fb in fallbacks:
                if not np.array_equal(fb, att1):
                    att2 = fb
                    break
            if att2 is None:
                att2 = t_in.copy()

        results.append({
            'attempt_1': n2g(att1),
            'attempt_2': n2g(att2),
        })

    return results


print('Solver defined.')

## 5. Inference Loop

In [ ]:
submission = {}
rule_stats = {}          # track which rules fire most often
t_global   = time.time()

for i, (task_id, task_data) in enumerate(challenges.items()):
    submission[task_id] = solve_task(task_data)

elapsed = time.time() - t_global
print(f'Solved {len(submission):,} tasks in {elapsed:.2f} s  '
      f'({elapsed/max(len(submission),1)*1000:.1f} ms/task avg)')

## 6. Schema Validation and Save

In [ ]:
assert len(submission) == len(challenges), 'Size mismatch!'
for k, v in submission.items():
    assert isinstance(v, list), f'{k}: must be list'
    for pred in v:
        assert 'attempt_1' in pred and 'attempt_2' in pred, f'{k}: missing attempt keys'
        assert isinstance(pred['attempt_1'], list)
        assert isinstance(pred['attempt_2'], list)
        # Ensure no empty grids
        assert len(pred['attempt_1']) > 0, f'{k}: empty attempt_1'
        assert len(pred['attempt_2']) > 0, f'{k}: empty attempt_2'

out_dir  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../submissions')
out_dir.mkdir(parents=True, exist_ok=True)
sub_path = out_dir / 'submission.json'

with open(sub_path, 'w') as f:
    json.dump(submission, f)

print(f'✅ Submission saved → {sub_path}')
print(f'   Tasks : {len(submission):,}')
print(f'   Size  : {sub_path.stat().st_size / 1024:.2f} KB')